
# 🔧 Simple Tool Selection & Tool Calling with LangChain

### Dinesh AI Academy | Day 3

## Goal

Build the smallest possible LangChain example that clearly shows:

```text
User Question
     ↓
LLM sees available tools
     ↓
LLM selects a tool
     ↓
LLM creates tool arguments
     ↓
Application executes the tool
     ↓
Tool result
```

### Important distinction

> **The LLM selects/requests the tool. LangChain/application executes the tool.**

We will use only **two tools**:

- `add_numbers`
- `get_weather`

This notebook intentionally avoids agents, LangGraph, RAG, memory, and complex orchestration.



## 1. Install packages

We need LangChain and the Google Gemini integration.


In [1]:
!pip -q install -U langchain langchain-google-genai


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



## 2. Configure Gemini

Enter your Gemini API key.

> Keep API keys private. Never commit them to GitHub.


In [1]:

import os
from getpass import getpass

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

print("✅ API key loaded successfully.")

os.environ["GOOGLE_API_KEY"] = GAISTUDIO_API_KEY

print("API key configured.")


✅ API key loaded successfully.
API key configured.



## 3. Create two simple tools

A LangChain tool is simply a function exposed to the LLM with a clear name and description.

The **description is important** because it tells the model what the tool is capable of.


In [2]:

from langchain_core.tools import tool

@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. This is a demo tool."""
    return f"Demo weather for {city}: 28°C, partly cloudy"


tools = [add_numbers, get_weather]

for t in tools:
    print(f"Tool: {t.name}")
    print(f"Description: {t.description}")
    print()


Tool: add_numbers
Description: Add two numbers together.

Tool: get_weather
Description: Get the current weather for a city. This is a demo tool.




# 4. Create the Gemini model

Now we give Gemini access to the tools.

**We are not asking Gemini to execute them.**

We are telling Gemini:

> "These are capabilities available to you. If appropriate, request one."


In [5]:

from langchain_google_genai import ChatGoogleGenerativeAI

MODEL = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    temperature=0
)

llm_with_tools = llm.bind_tools(tools)

print("Gemini is ready with tools.")


Gemini is ready with tools.



# 5. Ask a question that needs a tool

Let's ask:

> **What is 25 + 75?**

Instead of immediately asking for a final natural-language answer, we inspect the AI message.

Look specifically at:

```python
response.tool_calls
```

This is the important part of the demonstration.


In [6]:

from langchain_core.messages import HumanMessage

question = "What is 25 + 75?"

response = llm_with_tools.invoke([
    HumanMessage(content=question)
])

print("AI RESPONSE:")
print(response)

print("\nTOOL CALLS:")
print(response.tool_calls)


c:\Dinesh AI Academy\git_repos\5_day_ai_bootcamp\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI RESPONSE:
content=[] additional_kwargs={'function_call': {'name': 'add_numbers', 'arguments': '{"b": 75, "a": 25}'}, '__gemini_function_call_thought_signatures__': {'call_252728': 'EvUCCvICARFNMg+qFcbXokeuXkPqepMTar8/dP91RJ/xo5I7bt1KMNUUbi3sIabPB/gXmDvtabEVp+hJpJA7Fg3W6i4PheEWmbj01eMRO/rnxN1YObIMAaw4LsxxEGOL8kmLuvU0xKPmI7Y1PtrZtlvaiq0tV8ICGG+y3alts3oFPjZ9QbuaOytKsyy8VFoLNmugI+ALFoSWiSbPOfX0c9xzG2WOLkigdz2vAISgFHKjjggshdoYJnoK6hKGER/O+6HE+lJ09D5SW7e4b4w+Q4K2/HYvvQ+JF3J5wUan+go1aF5l+NHC4tequXXi7TUAfRuooaK+D+M3UaKHiVMdrtFjWDBoExDbnqLepWsflQVP6qV/I4ksj1dnADe3NLYTm1PEeLynzgRQ36mBnTVqqm5gZt3jJsTsQn2GirSKyCpmgKf7iIDMxwHe54fLxJ696V3wPIBwH/SQdmbmhPz5O9HKKJ5/mby4hyYnWCBLLN0DdFphFZSH2Q=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0a897-5fba-77c1-b2ad-3bd1207929e5-0' tool_calls=[{'name': 'add_numbers', 'args': {'b': 75, 'a': 25}, 'id': 'call_252728', 'type': 'tool_call'}] invalid_tool_ca


## What should we look for?

You should see something conceptually similar to:

```python
[
    {
        "name": "add_numbers",
        "args": {
            "a": 25,
            "b": 75
        },
        "id": "..."
    }
]
```

### This tells us three things

**1. Tool selected**

```text
add_numbers
```

**2. Arguments generated**

```text
a = 25
b = 75
```

**3. The tool has NOT necessarily been executed yet**

The model has generated a **tool call request**.

Our application still needs to execute it.



# 6. Execute the tool

Now LangChain/application code takes the model's tool call and invokes the actual Python function.

This is the key boundary:

```text
             LLM
              │
              │ tool call
              ▼
     ┌──────────────────┐
     │ LangChain / App  │
     └────────┬─────────┘
              │
              │ execute
              ▼
       add_numbers()
              │
              ▼
            100
```


In [7]:

tool_call = response.tool_calls[0]

selected_tool = next(
    tool for tool in tools
    if tool.name == tool_call["name"]
)

print("Selected tool:", selected_tool.name)
print("Arguments:", tool_call["args"])

result = selected_tool.invoke(tool_call["args"])

print("Tool result:", result)


Selected tool: add_numbers
Arguments: {'b': 75, 'a': 25}
Tool result: 100.0



# 7. Now try the weather tool

Change only the user question:

> **What is the weather in Delhi?**

The available tools are still:

```text
add_numbers
get_weather
```

The model should request `get_weather`.


In [8]:

question = "What is the weather in Delhi?"

response = llm_with_tools.invoke([
    HumanMessage(content=question)
])

print("USER:", question)
print()
print("TOOL CALLS:")
print(response.tool_calls)


c:\Dinesh AI Academy\git_repos\5_day_ai_bootcamp\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: What is the weather in Delhi?

TOOL CALLS:
[{'name': 'get_weather', 'args': {'city': 'Delhi'}, 'id': 'call_263754', 'type': 'tool_call'}]



# 8. One question — no tool required

Now ask:

> **What is RAG in AI?**

Neither `add_numbers` nor `get_weather` is useful.

The model can simply answer normally.

This demonstrates:

```text
User request
     ↓
Are available tools useful?
     ├── Yes → generate tool call
     └── No  → generate normal response
```

The model is not required to call a tool every time tools are available.


In [9]:

question = "What is RAG in AI?"

response = llm_with_tools.invoke([
    HumanMessage(content=question)
])

print("USER:", question)
print()
print("TOOL CALLS:", response.tool_calls)
print()
print("MODEL RESPONSE:")
print(response.content)


c:\Dinesh AI Academy\git_repos\5_day_ai_bootcamp\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: What is RAG in AI?

TOOL CALLS: []

MODEL RESPONSE:
[{'type': 'text', 'text': "**RAG** stands for **Retrieval-Augmented Generation**. It is a technique used in Artificial Intelligence, specifically with Large Language Models (LLMs), to improve the accuracy, relevance, and reliability of generated responses by incorporating external information sources.\n\n---\n\n### How RAG Works\n\nWithout RAG, a standard language model relies purely on its pre-trained knowledge base (what it learned during training up to a specific cutoff date).\n\nWith RAG, the system follows a three-step process:\n\n1. **Retrieval**: When a user submits a query, the system searches an external database, knowledge base, or document repository (e.g., company files, web pages, vector databases) for relevant context or documents.\n2. **Augmentation**: The retrieved information is appended to the user's original query as additional context within the prompt sent to the LLM.\n3. **Generation**: The LLM uses both it


# 9. See the selection process with three questions

Run these one after another:

```text
A. What is 50 + 30?
B. What is the weather in Mumbai?
C. Explain embeddings.
```

Same model.  
Same tools.  
Different user intent.

Observe the difference in `response.tool_calls`.


In [10]:

questions = [
    "What is 50 + 30?",
    "What is the weather in Mumbai?",
    "Explain embeddings in simple words."
]

for question in questions:
    response = llm_with_tools.invoke([
        HumanMessage(content=question)
    ])

    print("=" * 60)
    print("USER:", question)
    print("TOOL CALLS:", response.tool_calls)
    print("TEXT:", response.content)


c:\Dinesh AI Academy\git_repos\5_day_ai_bootcamp\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: What is 50 + 30?
TOOL CALLS: [{'name': 'add_numbers', 'args': {'b': 30, 'a': 50}, 'id': 'call_191832', 'type': 'tool_call'}]
TEXT: []


c:\Dinesh AI Academy\git_repos\5_day_ai_bootcamp\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: What is the weather in Mumbai?
TOOL CALLS: [{'name': 'get_weather', 'args': {'city': 'Mumbai'}, 'id': 'call_254127', 'type': 'tool_call'}]
TEXT: []


c:\Dinesh AI Academy\git_repos\5_day_ai_bootcamp\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: Explain embeddings in simple words.
TOOL CALLS: []
TEXT: [{'type': 'text', 'text': 'In simple terms, an **embedding** is a way of turning words, images, or ideas into a list of numbers so that computers can understand what they mean and how they relate to each other.\n\n---\n\n### Think of it like a map\n\nImagine a map where every word has its own location:\n\n* Words with **similar meanings** are placed **close together** (like *"cat"* and *"dog"*, or *"happy"* and *"joyful"*).\n* Words with **different meanings** are placed **far apart** (like *"cat"* and *"airplane"*).\n\nAn **embedding** is just the set of coordinates (the numbers) that tell the computer where each word belongs on that map.\n\n---\n\n### A Simple Example: Describing Animals\n\nSuppose you want a computer to understand animals using three traits (numbers from 0 to 10):\n1. **Size** (0 = tiny, 10 = huge)\n2. **Fluffiness** (0 = smooth, 10 = super fluffy)\n3. **Friendliness as a pet** (0 = dangerous, 10 = great


# 10. The complete mental model

This is the most important diagram for students:

```text
┌──────────────────────┐
│ User                 │
│ "What is 25 + 75?"   │
└──────────┬───────────┘
           │
           ▼
┌──────────────────────────────┐
│ Gemini + available tools     │
│                              │
│ add_numbers(a, b)            │
│ get_weather(city)            │
└──────────────┬───────────────┘
               │
               ▼
        ┌─────────────┐
        │    LLM      │
        └──────┬──────┘
               │
               │ generated tool call
               ▼
     ┌─────────────────────┐
     │ add_numbers         │
     │ a=25, b=75          │
     └──────────┬──────────┘
                │
                ▼
       ┌────────────────┐
       │ Python function│
       └───────┬────────┘
               │
               ▼
             100
```

### Remember

**Tool selection:** LLM generates the tool call.

**Tool calling/execution:** Application/LangChain invokes the actual tool.



# 11. Why does the model select a particular tool?

We give the model tool definitions such as:

```text
add_numbers
"Add two numbers together."

get_weather
"Get the current weather for a city."
```

For:

```text
"What is 25 + 75?"
```

the request is related to the capability described by `add_numbers`.

For:

```text
"What is the weather in Delhi?"
```

it is related to `get_weather`.

### Engineering explanation

```text
User intent
     +
Tool names
     +
Tool descriptions
     +
Tool parameter schemas
     +
Conversation context
     ↓
LLM computation
     ↓
Structured tool call
```

We can inspect the **generated tool call**, but we should not claim to see the model's private chain-of-thought.

Do **not** teach students that the model necessarily computes a visible numerical "score" for every tool unless the specific model/API documents such behavior.



# 12. Tool description experiment

Let's intentionally change the description of `add_numbers`.

Good description:

```text
"Add two numbers together."
```

A vague description:

```text
"Does something with numbers."
```

In real systems, tool descriptions should be:

- Clear
- Specific
- Unambiguous
- Focused on when the tool should be used
- Clear about parameters

Tool design directly affects the reliability of tool selection.



# 🎯 Final takeaway

### Tool

> **A capability the application exposes to the LLM.**

### Tool selection

> **The LLM generates a structured request for the tool it believes can help with the user's request.**

### Tool execution

> **The application/LangChain executes that requested tool and obtains the result.**

### The complete loop

```text
USER
 ↓
LLM
 ↓
TOOL CALL
 ↓
LANGCHAIN / APPLICATION
 ↓
TOOL EXECUTION
 ↓
TOOL RESULT
 ↓
LLM
 ↓
FINAL ANSWER
```

## One sentence students should remember

> **The LLM decides what tool call to request; LangChain/application executes the tool and returns the result.**

### Next concept

```text
Tool Calling
     ↓
Multiple Tools
     ↓
Workflow
     ↓
Agent
     ↓
Multi-step Agent
     ↓
MCP
```



## Instructor tip

For the live class, stop after each of these three questions:

```text
"What is 50 + 30?"
"What is the weather in Mumbai?"
"Explain embeddings."
```

Ask students:

> **"Did we change the tools?"**

No.

Then ask:

> **"What changed?"**

The **user's goal/request** changed.

Finally ask:

> **"What did Gemini return?"**

A normal answer when no tool is needed, or a **structured tool call** when a tool is useful.

That observation is the cleanest way to introduce tool calling before moving to workflows and agents.
